In [0]:
# CAMADA BRONZE - Ingestão direta do Volume
# Databricks notebook source
# COMMAND ----------
from pyspark.sql.functions import current_timestamp, lit

# Caminho base do Volume
volume_path = "/Volumes/mvp_eng_dados/bronze/base_dados"

# 1. Ingestão do Dataset Egressos direto do CSV no Volume
df_egressos_bronze_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{volume_path}/dataset_egressos_posgrad_erp.csv")
    .withColumn("_data_ingestao", current_timestamp())
    .withColumn("_arquivo_origem", lit(f"{volume_path}/dataset_egressos_posgrad_erp.csv"))
)

# 2. Ingestão do Dataset Cursos direto do CSV no Volume
df_cursos_bronze_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{volume_path}/dataset_cursos.csv")
    .withColumn("_data_ingestao", current_timestamp())
    .withColumn("_arquivo_origem", lit(f"{volume_path}/dataset_cursos.csv"))
)

# Criação do schema bronze se não existir
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp_eng_dados.bronze")

# Gravação nas tabelas Delta Bronze (com overwriteSchema para corrigir tipos)
df_egressos_bronze_raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_eng_dados.bronze.dataset_egressos_bruto")
df_cursos_bronze_raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_eng_dados.bronze.dataset_cursos_bruto")